### Configuration, inputs & outputs

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pipeline_utils import make_case3_paths, binarize_mutation_inclusive, matrix_stats

paths = make_case3_paths()

## Input files
batch_metrics_file = paths["datasets"] / "uhn_breast" / "batch_sensitivity.csv"
model_sensitivity_file = paths["datasets"] / "uhn_breast" / "model_sensitivity.csv"
expdesign_file = paths["datasets"] / "uhn_breast" / "expDesign.csv"
cnv_file = paths["datasets"] / "uhn_breast" / "CNV_matrix.csv"
mutation_file = paths["datasets"] / "uhn_breast" / "mutation_matrix.csv"
gene_file = paths["raw"] / "nest_vnn" / "gene2ind.txt"

## Output files
car_sensitivity_out = paths["proc"] / "UHN_car_sensitivity.csv"
models_out = paths["proc"] / "UHN_carboplatin_models.tsv"
mut_out = paths["proc"] / "UHN_mutation_carboplatin_filtered.csv"
cnv_out = paths["proc"] / "UHN_cnv_carboplatin_filtered.csv"

## NestVNN files (complete cohort)
nest_complete_dir = paths["proc"] / "UHN_nest_input" / "complete_cohort"
nest_complete_dir.mkdir(parents=True, exist_ok=True)
cell_vnn_complete = nest_complete_dir / "cell2ind.txt"
gene_vnn_complete = nest_complete_dir / "gene2ind.txt"
mut_vnn_complete = nest_complete_dir / "cell2mutation.txt"
cna_vnn_complete = nest_complete_dir / "cell2amplification.txt"
cnd_vnn_complete = nest_complete_dir / "cell2cndeletion.txt"
test_data_vnn_complete = nest_complete_dir / "test_data.txt"

## NestVNN files (strict cohort)
nest_strict_dir = paths["proc"] / "UHN_nest_input" / "strict_cohort"
nest_strict_dir.mkdir(parents=True, exist_ok=True)
cell_vnn_strict = nest_strict_dir / "cell2ind.txt"
gene_vnn_strict = nest_strict_dir / "gene2ind.txt"
mut_vnn_strict = nest_strict_dir / "cell2mutation.txt"
cna_vnn_strict = nest_strict_dir / "cell2amplification.txt"
cnd_vnn_strict = nest_strict_dir / "cell2cndeletion.txt"
test_data_vnn_strict = nest_strict_dir / "test_data.txt"

### Load response table and filter carboplatin models

In [2]:
batch_sen = pd.read_csv(batch_metrics_file)
model_sen = pd.read_csv(model_sensitivity_file)
exp_dsg = pd.read_csv(expdesign_file)
cnv = pd.read_csv(cnv_file, index_col=0)
mut = pd.read_csv(mutation_file, index_col=0)

treatment_arm = exp_dsg[exp_dsg["arm"] == "treatment"]

# n_replicates: number of treatment-arm mice recorded per batch in expDesign.csv
# (batch_sensitivity.csv has no n_replicates column of its own)
n_replicates_derived = (
    treatment_arm.groupby("batch.name").size().rename("n_replicates").reset_index()
)

# mRECIST: model_sensitivity.csv records it per individual mouse, not per batch.
# Aggregate to one value per batch via majority vote across that batch's
# treatment-arm mice; ties broken by worst response (PD > SD > PR > CR).
MRECIST_WORST_FIRST = ["PD", "SD", "PR", "CR"]

def majority_vote_mrecist(values):
    counts = values.dropna().value_counts()
    if counts.empty:
        return None
    top_count = counts.max()
    tied = counts[counts == top_count].index
    return next(r for r in MRECIST_WORST_FIRST if r in tied)

mouse_mrecist = treatment_arm.merge(model_sen[["model.id", "mRECIST"]], on="model.id", how="left")
mrecist_derived = (
    mouse_mrecist.groupby("batch.name")["mRECIST"]
    .apply(majority_vote_mrecist)
    .rename("mRECIST")
    .reset_index()
)

# batch.name contains "CARBOPLATIN"; exclude resistant-derivative lines (naming
# convention includes "-" or "RES", e.g. "REF001CARBORESP3.CARBOPLATIN")
car = batch_sen[
    batch_sen["batch.name"].str.contains("CARBOPLATIN")
    & ~batch_sen["batch.name"].str.contains("-")
    & ~batch_sen["batch.name"].str.contains("RES")
].reset_index(drop=True)

car["model.id"] = car["batch.name"].str.split(".").str[0]
car["drug.id"] = car["batch.name"].str.split(".").str[1]

car = car.merge(n_replicates_derived, on="batch.name", how="left")
car = car.merge(mrecist_derived, on="batch.name", how="left")

# Omics sample IDs are prefixed "S" for numeric model IDs (e.g. cnv/mut column
# "S108099"), but batch_metrics.csv stores the bare number ("108099")
mask = car["model.id"].str.match(r'^\d', na=False)
car.loc[mask, "model.id"] = "S" + car.loc[mask, "model.id"]

# Keep only models with CNV omics data available
n_before_omics_filter = car.shape[0]
car = car[car["model.id"].isin(cnv.columns)].reset_index(drop=True)
car = car.sort_values("model.id").reset_index(drop=True)
print(f"Carboplatin batches (non-resistant): {n_before_omics_filter}")
print(f"With CNV omics data available: {car.shape[0]}")

# For UHN, model.id already matches the CNV/mutation matrix column directly
# (no separate biobase-sample mapping is needed, unlike McGill) -- kept as its
# own column for schema parity with McGill_cisplatin_models.tsv.
car["omics_id"] = car["model.id"]

car.to_csv(car_sensitivity_out)
car["model.id"].to_csv(cell_vnn_complete, header=False, sep="\t")
print(f"\nTotal CARBOPLATIN models: {len(car)}")
print(car["mRECIST"].value_counts())

# A model's mRECIST can be missing even though it has CNV/mutation data -- this
# happens when none of its treatment-arm mice have a recorded mRECIST in
# model_sensitivity.csv (e.g. REF063 has exactly one treatment mouse, and that
# mouse's mRECIST is NaN). These models are kept in the cohort (they still have
# valid omics data), just flagged here since they're silently excluded from the
# mRECIST value_counts above.
missing_mrecist = car.loc[car["mRECIST"].isna(), "model.id"].tolist()
if missing_mrecist:
    print(f"\nModels kept but with no recorded mRECIST: {missing_mrecist}")

car.to_csv(models_out, sep="\t", index=False)
print(f"Saved model list to: {models_out}")

# Full (all-gene, raw/unbinarized) mutation and CNV matrices for every model in
# the complete cohort -- mirrors McGill_mutation_cisplatin_filtered.csv /
# McGill_cnv_cisplatin_filtered.csv. Not yet restricted to the 718-gene
# NeST-VNN panel or binarized (that happens later, in the gene-loading step).
mut_filtered = mut[mut.columns.intersection(car["model.id"])]
cnv_filtered = cnv[cnv.columns.intersection(car["model.id"])]
print(f"\nFiltered mutation matrix: {mut_filtered.shape}")
print(f"Filtered CNV matrix: {cnv_filtered.shape}")
mut_filtered.to_csv(mut_out)
cnv_filtered.to_csv(cnv_out)
print(f"Saved: {mut_out}")
print(f"Saved: {cnv_out}")

car_strict = car[car["n_replicates"] >= 2].reset_index(drop=True)
car_strict["model.id"].to_csv(cell_vnn_strict, header=False, sep="\t")
print(f"\nSTRICT cohort (n_replicates >= 2): {len(car_strict)}")
print(car_strict["mRECIST"].value_counts())

Carboplatin batches (non-resistant): 84
With CNV omics data available: 68

Total CARBOPLATIN models: 68
mRECIST
PD    34
SD    23
PR     6
CR     4
Name: count, dtype: int64

Models kept but with no recorded mRECIST: ['REF063']
Saved model list to: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/3_carboplatin/UHN_carboplatin_models.tsv

Filtered mutation matrix: (19725, 68)
Filtered CNV matrix: (25888, 68)


Saved: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/3_carboplatin/UHN_mutation_carboplatin_filtered.csv
Saved: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/3_carboplatin/UHN_cnv_carboplatin_filtered.csv

STRICT cohort (n_replicates >= 2): 36
mRECIST
PD    19
SD    12
CR     3
PR     2
Name: count, dtype: int64


### Report replicate condition

In [3]:
print(f"n_replicates distribution across the {len(car)} carboplatin models:")
print(car["n_replicates"].value_counts().sort_index())

n_multi_replicate = (car["n_replicates"] > 1).sum()
print(f"\nModels with >1 treatment-arm replicate: {n_multi_replicate}/{len(car)}")

n_replicates distribution across the 68 carboplatin models:
n_replicates
1    32
2    21
3     8
4     2
5     3
6     1
7     1
Name: count, dtype: int64

Models with >1 treatment-arm replicate: 36/68


### Inspect CNV matrix

In [4]:
print("CNV value range:", cnv.min().min(), "-", cnv.max().max())
cnv.head()

CNV value range: 0.0 - 186.0


,S108099,REF020,S45086,S71208,S100534,S53782,REF022,REF032,REF061,REF048,...,S73230,S66684,S100804,REF063,S28077,REF057,REF030,S44441,S69196,S90189
A1BG,4.0,4.0,1.0,3.0,3.0,1.0,1.0,3.0,4.0,1.0,...,4.0,3.0,NaN,NaN,3.0,NaN,3.0,NaN,1.0,1.0
A1BG-AS1,4.0,4.0,1.0,3.0,3.0,1.0,1.0,3.0,4.0,1.0,...,4.0,3.0,NaN,NaN,3.0,NaN,3.0,NaN,1.0,1.0
A1CF,3.0,NaN,6.0,4.0,NaN,3.0,NaN,4.0,NaN,NaN,...,3.0,3.0,NaN,NaN,5.0,NaN,3.0,3.0,NaN,3.0
A2M,4.0,3.0,3.0,NaN,5.0,3.0,3.0,1.0,NaN,NaN,...,5.0,5.0,NaN,3.0,5.0,NaN,5.0,NaN,3.0,1.0
A2M-AS1,4.0,3.0,3.0,NaN,5.0,3.0,3.0,1.0,NaN,NaN,...,5.0,5.0,NaN,3.0,5.0,NaN,5.0,NaN,3.0,1.0


In [5]:
gene_map = pd.read_csv(gene_file, sep="\t", header=None)
gene_map.to_csv(gene_vnn_complete, sep="\t", header=False, index=False)
gene_map.to_csv(gene_vnn_strict, sep="\t", header=False, index=False)

gene_map.columns = ["index", "gene"]
gene_list = gene_map["gene"].tolist()
print("NEST_VNN Gene:", len(gene_list))

mut_bin = mut.map(binarize_mutation_inclusive)

mut_complete = mut_bin.loc[mut_bin.index.isin(gene_list), mut_bin.columns.isin(car["model.id"])]
mut_strict = mut_bin.loc[mut_bin.index.isin(gene_list), mut_bin.columns.isin(car_strict["model.id"])]
cnv_complete = cnv.loc[cnv.index.isin(gene_list), cnv.columns.isin(car["model.id"])]
cnv_strict = cnv.loc[cnv.index.isin(gene_list), cnv.columns.isin(car_strict["model.id"])]

# Every carboplatin model that passed the CNV filter should also have mutation data
assert mut_complete.shape[1] == car.shape[0], "Some complete-cohort models are missing mutation data"
assert mut_strict.shape[1] == car_strict.shape[0], "Some strict-cohort models are missing mutation data"

# Per-sample baseline copy number (modal CN across the full genome, not just the
# 718-gene panel). UHN samples don't share one ploidy -- most cluster near-triploid
# (modal CN = 3), consistent with aneuploid/whole-genome-duplicated solid tumor
# biology, but some sit at 4-7. Amplification is defined relative to each sample's
# own baseline (see build_nest_vnn_cohort_inputs below) rather than one fixed
# absolute cutoff for the whole cohort.
cnv_baseline = cnv[cnv.columns.intersection(car["model.id"])].mode(axis=0, dropna=True).iloc[0]
print(f"\nPer-sample baseline copy number (modal, full genome): {cnv_baseline.value_counts().sort_index().to_dict()}")

print(f"Complete cohort -- Mutation: {mut_complete.shape} | CNV: {cnv_complete.shape}")
print(f"Strict cohort   -- Mutation: {mut_strict.shape} | CNV: {cnv_strict.shape}")

NEST_VNN Gene: 718



Per-sample baseline copy number (modal, full genome): {0.0: 3, 1.0: 8, 3.0: 42, 4.0: 5, 5.0: 7, 6.0: 2, 7.0: 1}
Complete cohort -- Mutation: (670, 68) | CNV: (711, 68)
Strict cohort   -- Mutation: (670, 36) | CNV: (711, 36)


### Build NeST-VNN inputs (complete and strict cohorts)

In [6]:
def build_nest_vnn_cohort_inputs(cnv_cohort, mut_cohort, car_cohort, baseline, mut_path, cna_path, cnd_path, test_path):
    """Threshold CNV into amplification/deletion calls, reorient to cell x gene
    format in car_cohort's model order, and save all NeST-VNN input files for
    one cohort (complete or strict)."""
    # Amplification is defined relative to each sample's own baseline ploidy
    # (its modal copy number across the full genome), not a fixed absolute cutoff:
    # UHN samples range from near-diploid to near-tetraploid/aneuploid, so a flat
    # cutoff would be too strict for high-ploidy samples and too loose for low-
    # ploidy ones. "Amplified" = CN >= 2x that sample's own baseline, floored at 4
    # for the few samples whose own baseline is 0/1 (more likely low tumor purity
    # than genuine near-haploid biology, so a literal 2x/4x threshold there would
    # be degenerate).
    thresh = (2 * baseline.reindex(cnv_cohort.columns)).clip(lower=4)
    cna = cnv_cohort.ge(thresh, axis=1).astype(int).reindex(gene_list).fillna(0)
    # Deletion: CN==0 is an unambiguous homozygous loss regardless of ploidy
    # (zero copies is zero copies), so this stays a flat rule, not ploidy-relative.
    cnd = (cnv_cohort == 0).astype(int).reindex(gene_list).fillna(0)
    mut = mut_cohort.reindex(gene_list).fillna(0)

    # Reorient to cell x gene, in the same model order as car_cohort
    mut = mut.T.astype(int).reindex(car_cohort["model.id"])
    cna = cna.T.astype(int).reindex(car_cohort["model.id"])
    cnd = cnd.T.astype(int).reindex(car_cohort["model.id"])

    mut.to_csv(mut_path, header=False, index=False)
    cna.to_csv(cna_path, header=False, index=False)
    cnd.to_csv(cnd_path, header=False, index=False)

    # test_data.txt: placeholder AUC (0), prediction-only run.
    # "drug"/"model" values aren't used for lookup by NeST-VNN's predict.py
    # (verified against ml_models/nest_vnn/src/util.py); only "dataset" and
    # the cell2id mapping matter for the actual predictions.
    test = pd.DataFrame({
        "cell": car_cohort["model.id"],
        "drug": "CARBOPLATIN",
        "test": 0,
        "model": "PDX",
    })
    test.to_csv(test_path, sep="\t", header=False, index=False)

    return mut, cna, cnd

### Complete cohort

In [7]:
mut_complete, cna_complete, cnd_complete = build_nest_vnn_cohort_inputs(
    cnv_complete, mut_complete, car, cnv_baseline,
    mut_vnn_complete, cna_vnn_complete, cnd_vnn_complete, test_data_vnn_complete,
)
print("Mutation matrix:", mut_complete.shape)
print("CNA matrix:", cna_complete.shape)
print("CND matrix:", cnd_complete.shape)

Mutation matrix: (68, 718)
CNA matrix: (68, 718)
CND matrix: (68, 718)


### Strict cohort

In [8]:
mut_strict, cna_strict, cnd_strict = build_nest_vnn_cohort_inputs(
    cnv_strict, mut_strict, car_strict, cnv_baseline,
    mut_vnn_strict, cna_vnn_strict, cnd_vnn_strict, test_data_vnn_strict,
)
print("Mutation matrix:", mut_strict.shape)
print("CNA matrix:", cna_strict.shape)
print("CND matrix:", cnd_strict.shape)

Mutation matrix: (36, 718)
CNA matrix: (36, 718)
CND matrix: (36, 718)


### Matrix statistics (QC)

In [9]:
matrix_stats("Mutation (complete)", mut_complete)
matrix_stats("CNA (complete)", cna_complete)
matrix_stats("CND (complete)", cnd_complete)

matrix_stats("Mutation (strict)", mut_strict)
matrix_stats("CNA (strict)", cna_strict)
matrix_stats("CND (strict)", cnd_strict)


Mutation (complete) statistics
----------------------------
Models: 68
Genes: 718
Total entries: 48824
Nonzero entries: 2223
Sparsity: 0.9545

Per model events
Median: 17.5
Mean: 32.69
Max: 174

Per gene events
Median: 2.0
Mean: 3.1
Max: 55

CNA (complete) statistics
----------------------------
Models: 68
Genes: 718
Total entries: 48824
Nonzero entries: 5241
Sparsity: 0.8927

Per model events
Median: 60.5
Mean: 77.07
Max: 333

Per gene events
Median: 5.0
Mean: 7.3
Max: 38

CND (complete) statistics
----------------------------
Models: 68
Genes: 718
Total entries: 48824
Nonzero entries: 2990
Sparsity: 0.9388

Per model events
Median: 31.0
Mean: 43.97
Max: 225

Per gene events
Median: 1.0
Mean: 4.16
Max: 66

Mutation (strict) statistics
----------------------------
Models: 36
Genes: 718
Total entries: 25848
Nonzero entries: 1141
Sparsity: 0.9559

Per model events
Median: 17.0
Mean: 31.69
Max: 174

Per gene events
Median: 1.0
Mean: 1.59
Max: 32

CNA (strict) statistics
-----------------